In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import TensorBoard
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import os
import datetime
import json
import joblib

y:\Anaconda\envs\SAT5114\lib\site-packages\sklearn\utils\validation.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LARGE_SPARSE_SUPPORTED = LooseVersion(scipy_version) >= '0.14.0'


In [ ]:
os.makedirs("results", exist_ok=True)
os.makedirs("logs/fit/Dense/", exist_ok=True)

log_dir = "logs/fit/Dense/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [3]:
df = pd.read_csv("day_approach_maskedID_timeseries.csv")  
df = df.drop(columns=["Athlete ID", "Date"])
df.fillna(df.median(), inplace=True)

In [5]:
X = df.drop(columns=["injury"])
y = df["injury"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


y:\Anaconda\envs\SAT5114\lib\site-packages\sklearn\model_selection\_split.py:1609: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  return floored.astype(np.int)
y:\Anaconda\envs\SAT5114\lib\site-packages\sklearn\model_selection\_split.py:1609: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current u

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
y_train = np.array(y_train)
y_test = np.array(y_test)

In [8]:
class_weights = class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

In [9]:
X_train.shape[1]

70

In [11]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)), 
    layers.Dense(128, activation="relu"),  
    layers.Dropout(0.3),  
    layers.Dense(64, activation="relu"), 
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),  
    layers.Dense(1, activation="sigmoid")  
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
history = model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=32, 
    validation_data=(X_test, y_test), 
    class_weight=class_weight_dict,
    callbacks=[tensorboard_callback],  
    verbose=1
)

model.save("results/Dense/injury_prediction_model.h5")

Epoch 1/50
1070/1070 [==============================] - 2s 2ms/step - loss: 0.7156 - accuracy: 0.5550 - val_loss: 0.4839 - val_accuracy: 0.8749
Epoch 2/50
1070/1070 [==============================] - 1s 1ms/step - loss: 0.6791 - accuracy: 0.5189 - val_loss: 0.7221 - val_accuracy: 0.4514
Epoch 3/50
1070/1070 [==============================] - 1s 1ms/step - loss: 0.6600 - accuracy: 0.5499 - val_loss: 0.8209 - val_accuracy: 0.3450
Epoch 4/50
1070/1070 [==============================] - 2s 2ms/step - loss: 0.6447 - accuracy: 0.5576 - val_loss: 0.6232 - val_accuracy: 0.5908
Epoch 5/50
1070/1070 [==============================] - 1s 1ms/step - loss: 0.6399 - accuracy: 0.5694 - val_loss: 0.6691 - val_accuracy: 0.5228
Epoch 6/50
1070/1070 [==============================] - 2s 1ms/step - loss: 0.6292 - accuracy: 0.5780 - val_loss: 0.5275 - val_accuracy: 0.6703
Epoch 7/50
1070/1070 [==============================] - 1s 1ms/step - loss: 0.6275 - accuracy: 0.6011 - val_loss: 0.5782 - val_accuracy:

In [ ]:
with open("results/Dense/training_history.json", "w") as f:
    json.dump(history.history, f)

joblib.dump(scaler, "results/Dense/scaler.pkl")

with open("results/Dense/class_weights.json", "w") as f:
    json.dump(class_weight_dict, f)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred_prob))
print("Classification Report:\n", classification_report(y_test, y_pred))

class CustomTensorBoardCallback(TensorBoard):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs["roc_auc"] = roc_auc_score(y_test, self.model.predict(X_test))
        super().on_epoch_end(epoch, logs)

custom_tensorboard_callback = CustomTensorBoardCallback(log_dir=log_dir, histogram_freq=1)


268/268 [==============================] - 0s 578us/step
Accuracy: 0.8946691606266074
AUC-ROC: 0.6087340155136766
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94      8437
           1       0.02      0.17      0.04       117

   micro avg       0.89      0.89      0.89      8554
   macro avg       0.51      0.54      0.49      8554
weighted avg       0.97      0.89      0.93      8554

